In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(10)
random.seed(10)
n = 2000

# ---------- helper functions ----------
def random_ip():
    return ".".join(map(str, np.random.randint(0,255,4)))

def random_mac():
    return ":".join(["%02x" % random.randint(0,255) for _ in range(6)])

def random_time():
    start = datetime(2026,1,1)
    return start + timedelta(seconds=random.randint(0,500000))

# ---------- dataset ----------
data = {
    "Timestamp":[random_time() for _ in range(n)],
    "Source_IP":[random_ip() for _ in range(n)],
    "Destination_IP":[random_ip() for _ in range(n)],
    "Source_Port":np.random.randint(1024,65535,n),
    "Destination_Port":np.random.choice([21,22,53,80,443,8080],n),
    "Protocol":np.random.choice(["TCP","UDP","ICMP"],n),
    "Packet_Size":np.random.randint(40,2000,n),
    "Connection_Duration":np.random.randint(1,800,n),
    "Packets_Per_Second":np.random.randint(1,6000,n),
    "Error_Rate":np.random.random(n),
    "Login_Attempts":np.random.randint(0,25,n),
    "Bytes_Sent":np.random.randint(100,60000,n),
    "Bytes_Received":np.random.randint(100,70000,n),
    "Traffic_Flag":np.random.choice(["Normal","Suspicious","DoS","Probe"],n,p=[0.65,0.15,0.10,0.10]),  # realistic distribution
    "MAC_Address":[random_mac() for _ in range(n)]
}

df = pd.DataFrame(data)

# ---------- LABEL LOGIC ----------
# Label created BEFORE introducing missing values
df["Label"] = (
    ((df["Packet_Size"] > 1500) & (df["Error_Rate"] > 0.8)) |
    (df["Login_Attempts"] > 18) |
    ((df["Bytes_Sent"] > 40000) & (df["Traffic_Flag"] == "Suspicious")) |
    (df["Traffic_Flag"] == "DoS")
).astype(int)

# ---------- introduce realistic NULL values AFTER label ----------
df.loc[df.sample(frac=0.08).index,
       ["Packet_Size","Packets_Per_Second",
        "Bytes_Sent","Bytes_Received"]] = np.nan

df.loc[df.sample(frac=0.05).index,
       ["MAC_Address","Source_IP"]] = np.nan

df.loc[df.sample(frac=0.03).index, "Timestamp"] = np.nan

df.loc[df.sample(frac=0.06).index,
       ["Error_Rate","Connection_Duration"]] = np.nan

# ---------- save dataset as CSV File----------
df.to_csv("Network_log.csv", index=False)
print(df)
print(df["Label"].value_counts())

               Timestamp        Source_IP  Destination_IP  Source_Port  \
0    2026-01-03 01:08:40     9.125.228.15   57.94.114.108        31659   
1    2026-01-02 22:32:06   64.113.123.156   70.39.249.212         5860   
2    2026-01-03 22:16:02  217.221.240.157    25.38.159.82        58205   
3    2026-01-04 11:41:18     113.250.8.73  98.218.235.251         7700   
4    2026-01-05 16:19:21     0.234.40.246   80.230.239.28        27178   
...                  ...              ...             ...          ...   
1995 2026-01-02 15:44:51  246.182.214.222    53.75.137.23        48852   
1996 2026-01-03 10:17:42              NaN     97.91.97.19        59758   
1997 2026-01-02 12:04:25    81.141.132.60   26.243.229.86        45616   
1998 2026-01-06 11:20:49     40.215.55.67   80.56.249.249        50368   
1999 2026-01-01 19:40:10   209.21.190.247   91.116.100.77        16554   

      Destination_Port Protocol  Packet_Size  Connection_Duration  \
0                   80      UDP       1461